# ABLE Distillation: Local Fine-Tune Qwen/Qwen3.5-9B

Generated: 2026-04-11 21:46 UTC

**Model**: Qwen/Qwen3.5-9B (edge)
**Corpus**: `data/corpus/default/v048/train.jsonl`
**Epochs**: 3

Auto-detects hardware: **CUDA** (Unsloth 2x faster) | **MPS** (PEFT LoRA) | **CPU** (testing only)

Run in VS Code Jupyter, JupyterLab, or any local notebook environment.

In [ ]:
MODEL_NAME = "able-nano-9b"
BASE_MODEL = "Qwen/Qwen3.5-9B"
CORPUS_PATH = "data/corpus/default/v048/train.jsonl"
EPOCHS = 3
LEARNING_RATE = 0.0002
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 16
IS_GEMMA4 = False
QUANT_METHODS = ['q4_k_m', 'iq2_m', 'q8_0']

In [ ]:
import sys, platform, subprocess, torch
from pathlib import Path

DEVICE = 'cpu'
USE_UNSLOTH = False
DTYPE = 'float32'
MEMORY_GB = 0.0

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    DTYPE = 'float16'
    try:
        r = subprocess.run(['sysctl', '-n', 'hw.memsize'],
                           capture_output=True, text=True, timeout=5)
        MEMORY_GB = round(int(r.stdout.strip()) / (1024**3), 1)
    except Exception:
        MEMORY_GB = 16.0
elif torch.cuda.is_available():
    DEVICE = 'cuda'
    MEMORY_GB = round(torch.cuda.get_device_properties(0).total_mem / (1024**3), 1)
    cap = torch.cuda.get_device_capability(0)
    DTYPE = 'bfloat16' if cap[0] >= 8 else 'float16'
    USE_UNSLOTH = True

# Auto batch sizing
if MEMORY_GB >= 64: BATCH_SIZE, GRAD_ACCUM = 8, 2
elif MEMORY_GB >= 32: BATCH_SIZE, GRAD_ACCUM = 4, 4
elif MEMORY_GB >= 16: BATCH_SIZE, GRAD_ACCUM = 2, 8
else: BATCH_SIZE, GRAD_ACCUM = 1, 16

print(f'Device: {DEVICE} | Memory: {MEMORY_GB}GB | '
      f'Unsloth: {USE_UNSLOTH} | Batch: {BATCH_SIZE}x{GRAD_ACCUM}')

In [ ]:
import subprocess, sys, os
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

if USE_UNSLOTH:
    pip_install('unsloth>=2026.4.3')
    pip_install('--no-deps', 'trl', 'peft', 'accelerate', 'bitsandbytes')
else:
    pip_install('torch', 'transformers', 'peft', 'trl', 'accelerate', 'datasets')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
print('Dependencies installed.')

In [ ]:
import torch

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(model, r=LORA_R,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=LORA_ALPHA, lora_dropout=0,
        use_gradient_checkpointing='unsloth')
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    load_dtype = torch.float16 if DEVICE == 'mps' else torch.float32
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL,
        torch_dtype=load_dtype, attn_implementation='eager', low_cpu_mem_usage=True)
    if DEVICE == 'mps': model = model.to('mps')
    model.gradient_checkpointing_enable()
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0, task_type='CAUSAL_LM')
    model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Model: {BASE_MODEL} — {trainable:,}/{total:,} trainable ({trainable/total*100:.2f}%)')

In [ ]:
from datasets import load_dataset
from pathlib import Path

corpus = Path(CORPUS_PATH)
if not corpus.exists(): corpus = Path('../') / CORPUS_PATH
if not corpus.exists(): raise FileNotFoundError(f'Corpus not found: {CORPUS_PATH}')

def format_chatml(example):
    messages = example.get('messages', example.get('conversations', []))
    return {'text': tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)}

dataset = load_dataset('json', data_files=str(corpus), split='train')
dataset = dataset.map(format_chatml)
print(f'Loaded {len(dataset)} training examples')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_kwargs = dict(
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    logging_steps=10, save_strategy='steps', save_steps=100,
    output_dir=f'outputs/able-nano-9b', warmup_steps=5,
    weight_decay=0.01, lr_scheduler_type='linear', seed=42, report_to='none')

if USE_UNSLOTH:
    from unsloth import is_bfloat16_supported
    training_kwargs.update(fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(), optim='adamw_8bit')
elif DEVICE == 'mps':
    training_kwargs.update(fp16=False, bf16=False,
        dataloader_pin_memory=False, optim='adamw_torch')
else:
    training_kwargs.update(fp16=False, bf16=False, no_cuda=True, optim='adamw_torch')

trainer = SFTTrainer(model=model, tokenizer=tokenizer,
    train_dataset=dataset, dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2,
    args=TrainingArguments(**training_kwargs))

trainer_stats = trainer.train()
print(f'Training complete. Loss: {trainer_stats.training_loss:.4f}')

## Export to GGUF for Ollama

In [ ]:
import os
os.makedirs(f'outputs/able-nano-9b-gguf', exist_ok=True)

if USE_UNSLOTH:
    for quant in ["q4_k_m", "iq2_m", "q8_0"]:
        print(f'Exporting {quant}...')
        model.save_pretrained_gguf(f'outputs/able-nano-9b-gguf',
            tokenizer, quantization_method=quant)
else:
    merged = model.merge_and_unload()
    merged.save_pretrained(f'outputs/able-nano-9b-merged')
    tokenizer.save_pretrained(f'outputs/able-nano-9b-merged')
    print(f'Merged model saved. Convert to GGUF with llama.cpp:')
    print(f'  python3 llama.cpp/convert_hf_to_gguf.py outputs/able-nano-9b-merged \\')
    print(f'    --outfile outputs/able-nano-9b-gguf/able-nano-9b-f16.gguf --outtype f16')

In [ ]:
import json
from datetime import datetime, timezone

stats = {
    'model': 'able-nano-9b', 'base': BASE_MODEL,
    'corpus_size': len(dataset), 'epochs': EPOCHS,
    'device': DEVICE, 'backend': 'unsloth' if USE_UNSLOTH else 'peft',
    'loss': trainer_stats.training_loss,
    'completed_at': datetime.now(timezone.utc).isoformat(),
}
with open(f'outputs/able-nano-9b_training_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))